
## Transform circuits data
1. Read bronze _`races`_ table
2. Keep only columns that are required (remove url col)
3. Standardize column names using snake_case
4. Rename columns to make them more meaningful ( date -> races_date)
5. Filter out rows where season and round is null
6. Remove duplicate rows
7. Transform values of race_name and to Title Case 
8. Write the tranformed data to silver table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


**1. Read bronze races table**

In [0]:
# circuits_df = spark.read.option('VersionAsOf',0).table(bronze_table)
# use spark.read.table if you need additional options, otherwise use spark.table for simplicity

In [0]:
races_df = spark.table(bronze_table)

**2. Keep only columns that are required (remove url col)**


In [0]:
# circuits_selected_df = circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "Ingestion_timestamps",
#     "source_file"
# )

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("Ingestion_timestamps"),
    F.col("source_file")
)

**Steps 3 and 4  Standardize column names**
- Standardize column names using snake_case 
- Rename columns to make them more meaningful 


In [0]:

# if you want to use withColumnRenamed it should be passed in indivdually .withColumnRenamed("circuitId","circuit_id") four times

In [0]:
races_renamed_df= (races_selected_df.withColumnsRenamed(
    {"circuitId": "circuit_id", 
     "raceName":"race_name"
     #"sourcefile":"source_file"
      })
    
    )

In [0]:
races_renamed_df.filter(F.col('season').isNull()).count()



**6. Remove duplicate rows**

In [0]:
#circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(['season', 'round'])

**7. Transform values of _circuit_id_ and race_name to Title Case**


In [0]:
races_final_df = (
    races_distinct_df
    # .withColumn("circuit_id", F.initcap(F.col("circuit_id")))
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

**8. Write the tranformed data to silver table**

In [0]:
(races_final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)
 
 